# 20 · A low and a high QP model, each told the QP

Notebook 18's one told model, against two: one for low QP and one for high, each still told the QP. Each half gets the
whole body for its own job, sharpening a clean frame or deblocking a coded one, at the price of training on part of the
data, shipping two sets of weights, and a switch between them.

- **The switch is at QP 30.**
- **The training ranges overlap**: the low model trains and selects on slice QP under 34, the high one on 26 and up, so
  each has seen frames on the far side of the switch.
- **The bar**: in each half, the split has to beat the one told model by clearly more than the one model's two seeds
  land apart.
- **The switch itself**: how much the picture changes when a stream's QP crosses 30, against how much one QP step
  changes it in the one model. A large jump would show as a flicker when bandwidth moves around the switch.

The same body (24 channels, 2 blocks) and recipe (`training.fit`) for every model, two seeds each; scored on notebook
0's validation split.

Needs: `18_upscale_qp_models.ipynb` (the one told model at two seeds, and its scores).

In [1]:
"""Settings."""

import os

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import metrics, models, patches, training
from helpers.training import DATA_DIR, DEVICE

NOTEBOOK = 20
WIDTH, BLOCKS, STEM = 24, 2, 3               # notebook 7's body
QP_MAX = patches.QP_MAX
SWITCH = 30                                  # the low model below it, the high one from it
OVERLAP = 4                                  # each model also trains this far past the switch
RANGES = {"low": (0, SWITCH + OVERLAP), "high": (SWITCH - OVERLAP, 99)}   # slice QP each model trains on
SEEDS = (0, 1)
TOLD = ("qp",)
COLUMNS = patches.columns(TOLD)
RUN_DIR = training.CHECKPOINT_DIR / "qp_runs"
ONE = {seed: RUN_DIR / f"told_qp_seed{seed}.pt" for seed in SEEDS}   # notebook 18's one told model

manifest = training.load_manifest()
SCALE = manifest["scale"]
recorded = training.recorded()
bicubic = recorded[1, "bicubic"]


def build(seed=0):
    torch.manual_seed(seed)
    return models.ToldLuma(len(COLUMNS), WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)


def run_file(part, seed):
    low, high = RANGES[part]
    return RUN_DIR / f"told_{part}_qp{low}-{high}_seed{seed}.pt"

## 1. Train

`training.fit` for each half at each seed, trained and selected on its own range only. The one told model is notebook
18's, loaded.

In [2]:
halves = {}
for seed in SEEDS:
    for part, qp_range in RANGES.items():
        model = build(seed)
        history, best = training.fit(model, run_file(part, seed), manifest, told=TOLD, seed=seed, qp_range=qp_range,
                                     verbose=0)
        halves[part, seed] = model.eval()
        print(f"{part} QP {qp_range[0]}-{qp_range[1]}, seed {seed}: best epoch {best['epoch']:3} of {len(history):3}, "
              f"select SSIM-Y {best['score']:.4f} on its range")

one = {}
for seed, path in ONE.items():
    one[seed] = build(seed)
    one[seed].load_state_dict(training.fitted_weights(path))
    one[seed].eval()

train   86533 patches at slice QP 0-34
select   4405 patches at slice QP 0-34, to choose the best epoch
low QP 0-34, seed 0: best epoch 119 of 131, select SSIM-Y 0.9321 on its range
train   96963 patches at slice QP 26-99
select   4691 patches at slice QP 26-99, to choose the best epoch
high QP 26-99, seed 0: best epoch 154 of 160, select SSIM-Y 0.8630 on its range
train   86533 patches at slice QP 0-34
select   4405 patches at slice QP 0-34, to choose the best epoch
low QP 0-34, seed 1: best epoch 112 of 118, select SSIM-Y 0.9323 on its range
train   96963 patches at slice QP 26-99
select   4691 patches at slice QP 26-99, to choose the best epoch
high QP 26-99, seed 1: best epoch 139 of 151, select SSIM-Y 0.8626 on its range


## 2. Gain over bicubic

The split as a client would run it: each patch through the low model if its slice QP is under 30, else through the
high one. Beside it the one told model at both seeds, overall and by slice QP bin; then by bin, the split's lead over
the one model (the mean of two seeds each) and how far apart each one's two seeds land.

In [3]:
def split(seed):
    low, high = halves["low", seed], halves["high", seed]

    def upscale(lr, condition):
        told = condition[:, COLUMNS]
        is_high = (told * QP_MAX).round() >= SWITCH   # (N, 1, 1, 1)
        return torch.where(is_high, high(lr, told), low(lr, told))
    return upscale


scores = {}
for seed in SEEDS:
    scores[f"split at QP {SWITCH}" + (f", seed {seed}" if seed else "")] = training.val_scores(split(seed), manifest)
for name, s in scores.items():
    training.record(NOTEBOOK, f"told the QP, {name}", s)
ones = {name: s for (notebook, name), s in recorded.items() if notebook == 18 and name.startswith("told the QP")}
training.show(training.compare({"bicubic": bicubic, **ones, **scores}, manifest, base=bicubic))
print()
training.show(training.gain_by_qp({**ones, **scores}, manifest, base=bicubic))

qp = np.load(DATA_DIR / manifest["splits"]["val"]["qp"])
edges = manifest["qp_edges"]
split_runs, one_runs = list(scores.values()), list(ones.values())
print(f"\n{'QP':>7}{'split - one':>14}{'one seeds':>11}{'split seeds':>13}   SSIM-Y")
for low, high in [(0, 99), *zip(edges[:-1], edges[1:])]:
    inside = (qp >= low) & (qp < high)
    mean_in = lambda s: s["ssim"][inside].mean()
    lead = np.mean([mean_in(s) for s in split_runs]) - np.mean([mean_in(s) for s in one_runs])
    print(f"{'all' if high == 99 else f'{low}-{high}':>7}{lead:+14.4f}{np.ptp([mean_in(s) for s in one_runs]):11.4f}"
          f"{np.ptp([mean_in(s) for s in split_runs]):13.4f}")

                 model SSIM-Y over bicubic          95% range luma dB dB over chroma dB over   video  screen   photo    x264   nvenc
        split at QP 30 0.8926      +0.0405 +0.0381 to +0.0429   30.31   +1.72          +0.05 +0.0236 +0.0595 +0.0259 +0.0428 +0.0382
split at QP 30, seed 1 0.8924      +0.0403 +0.0380 to +0.0428   30.30   +1.72          +0.05 +0.0233 +0.0594 +0.0257 +0.0426 +0.0381
           told the QP 0.8914      +0.0393 +0.0371 to +0.0417   30.21   +1.63          +0.05 +0.0230 +0.0578 +0.0249 +0.0417 +0.0370
   told the QP, seed 1 0.8909      +0.0388 +0.0365 to +0.0412   30.17   +1.59          +0.04 +0.0226 +0.0573 +0.0242 +0.0410 +0.0367
               bicubic 0.8521      +0.0000 +0.0000 to +0.0000   28.58   +0.00          +0.00 +0.0000 +0.0000 +0.0000 +0.0000 +0.0000

                 model    0-18   18-26   26-30   30-34   34-38   38-42   42-46   46-52
               patches    3610    3928    3730    3585    3539    2846    2005    1043
   told the QP, seed 1 +0.0

## 3. The switch

On the validation patches either side of the switch (slice QP 26 to 34), each patch through both halves and through
the one model told the QP one step apart:

- **split, low vs high**: the mean and the 99th percentile of the luma difference between the low and the high model's
  picture of the same patch, in 8-bit levels. This is what changes on screen when a stream's QP crosses 30.
- **one, QP vs QP + 1**: the same for the one model told the patch's QP and one more. This is what one QP step changes
  without a switch.

In [4]:
trim = training.trimmer(manifest)
near = np.flatnonzero((qp >= SWITCH - OVERLAP) & (qp < SWITCH + OVERLAP))
near_loader = DataLoader(Subset(patches.PatchPairs(DATA_DIR, "val", manifest, augment=False, told=TOLD), near),
                         batch_size=256)


def luma_255(x):
    return metrics.rgb_to_yuv(trim(x).clamp(0, 1))[:, :1] * 255


@torch.no_grad()
def differences(first, second):
    """Per-pixel luma differences (8-bit levels) between two models' pictures of the patches near the switch."""
    out = []
    for (lr, condition), _ in near_loader:
        lr, condition = lr.to(DEVICE), condition.to(DEVICE)
        out.append((luma_255(first(lr, condition)) - luma_255(second(lr, condition))).abs().flatten().cpu())
    return torch.cat(out)


print(f"luma change, slice QP {SWITCH - OVERLAP}-{SWITCH + OVERLAP} ({len(near)} patches), 8-bit levels")
print(f"{'':<28}{'mean':>8}{'99th %':>8}")
for seed in SEEDS:
    pairs = {f"split s{seed}, low vs high": (halves["low", seed], halves["high", seed]),
             f"one s{seed}, QP vs QP + 1": (one[seed], lambda x, c, model=one[seed]: model(x, c + 1 / QP_MAX))}
    for name, (first, second) in pairs.items():
        change = differences(first, second)
        print(f"{name:<28}{change.mean():8.2f}{np.percentile(change.numpy(), 99):8.2f}")

luma change, slice QP 26-34 (7315 patches), 8-bit levels
                                mean  99th %
split s0, low vs high           1.62   20.87
one s0, QP vs QP + 1            0.15    1.35
split s1, low vs high           1.67   21.75
one s1, QP vs QP + 1            0.14    1.20


## 4. What it says

- **Two told models beat one.** Switched at slice QP 30, a low and a high model, each told the QP, score 0.8926 and
  0.8924 at two seeds (+0.0405 and +0.0403 over bicubic): +0.0013 over notebook 18's one told model, whose two seeds
  land 0.0005 apart. That makes the split the best on this set, at the same body and multiply-adds; the price is a second
  set of weights and the switch.
- **It leads in every QP bin but the one past the switch**: +0.0024 at QP 0–18, +0.0015 to +0.0016 from 38 to 46, and
  +0.0002 from 30 to 34, just past the switch, inside the seeds' spread there.
- **The switch is a visible jump.** On patches at QP 26 to 34, the low and the high model's pictures of the same patch
  differ by 1.6 8-bit levels of luma on average and 21 at the 99th percentile; one QP step in the one model moves it
  0.15 and 1.3. A stream whose QP moves across 30 would flicker where the two disagree, which the scores here do not
  see. A blend of the two halves over a few QP steps is the untested way around it.
- **Better than a specialist.** Notebook 18's specialist, switched in at QP 38, would add +0.0005 overall; the split
  adds +0.0013, because its low half gains too (+0.0017 to +0.0024 below QP 26).